# 1. FAQ preprocessing

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


## 1.1. Mounting Google Drive

I am mounting Google Drive so that this notebook can access the ADNI non-imaging source files and save the processed FAQ and QC outputs directly to the existing project directory. Mounting Drive does not modify any source files.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1.2. Defining the FAQ project paths

I am defining the input and output paths used in this notebook. verify that the raw FAQ file and ADNI data dictionary exist before loading them. I am also creating the processed and QC directories if they are missing.

This step only prepares file paths and folders. It does not read, modify, or overwrite the raw CSV files.

In [ ]:
from pathlib import Path

# Define the root directory for the ADNI non-imaging project.
non_imaging_root = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

# Define the raw input files.
faq_path = (
    non_imaging_root
    / "raw"
    / "Cognitive and functional assessments"
    / "All_Subjects_FAQ_11Jul2026.csv"
)

datadic_path = (
    non_imaging_root
    / "raw"
    / "Cohort, dates and source-of-truth tables"
    / "DATADIC_11Jul2026.csv"
)

# Define the output directories.
processed_dir = non_imaging_root / "processed"
qc_dir = non_imaging_root / "qc"

# Create output directories if they do not already exist.
processed_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

# Define the planned output files without creating them yet.
faq_clean_output_path = (
    processed_dir / "faq_clean_longitudinal_interim.csv"
)

faq_exclusion_log_path = (
    qc_dir / "faq_exclusion_log.csv"
)

faq_qc_summary_path = (
    qc_dir / "faq_qc_summary.csv"
)

# Verify the project paths.
print("FAQ source file:")
print(f"  {faq_path}")
print(f"  Exists: {faq_path.exists()}")

print("\nADNI data dictionary:")
print(f"  {datadic_path}")
print(f"  Exists: {datadic_path.exists()}")

print("\nOutput directories:")
print(f"  Processed directory exists: {processed_dir.exists()}")
print(f"  QC directory exists: {qc_dir.exists()}")

print("\nPlanned outputs:")
print(f"  Clean FAQ table: {faq_clean_output_path}")
print(f"  Exclusion log:   {faq_exclusion_log_path}")
print(f"  QC summary:      {faq_qc_summary_path}")

## 1.3. Loading and inspecting the raw FAQ table

I am loading the FAQ CSV into an untouched dataframe named `faq_raw`. inspect its size, sample records, column names, data types, participant coverage, ADNI phase coverage, and visit-date range.

At this stage, I am only examining the source data. I am not changing values, dropping columns, removing rows, or selecting baseline visits.

In [ ]:
import pandas as pd

# Load the raw FAQ source table without modifying the original CSV.
faq_raw = pd.read_csv(faq_path, low_memory=False)

# Display the overall dimensions.
print("FAQ table dimensions:")
print(f"  Rows:    {faq_raw.shape[0]:,}")
print(f"  Columns: {faq_raw.shape[1]:,}")

# Display a small sample of records.
print("\nSample FAQ records:")
display(faq_raw.head())

# Print every column in its original order.
print("\nFAQ columns:")
for position, column in enumerate(faq_raw.columns, start=1):
    print(f"{position:>3}. {column}")

# Inspect inferred pandas data types.
print("\nData types:")
display(
    faq_raw.dtypes
    .rename("dtype")
    .to_frame()
)

# Count unique participants using RID.
if "RID" in faq_raw.columns:
    unique_rids = faq_raw["RID"].nunique(dropna=True)
    missing_rids = faq_raw["RID"].isna().sum()

    print("\nParticipant coverage:")
    print(f"  Unique non-missing RIDs: {unique_rids:,}")
    print(f"  Rows with missing RID:   {missing_rids:,}")
else:
    print("\nRID was not found in the FAQ table.")

# Inspect ADNI phase coverage.
if "PHASE" in faq_raw.columns:
    print("\nRows by ADNI phase:")
    phase_summary = (
        faq_raw["PHASE"]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .rename_axis("PHASE")
        .reset_index(name="row_count")
    )

    phase_summary["percentage"] = (
        phase_summary["row_count"] / len(faq_raw) * 100
    ).round(2)

    display(phase_summary)
else:
    print("\nPHASE was not found in the FAQ table.")

# Inspect the visit-date range without changing faq_raw.
if "VISDATE" in faq_raw.columns:
    faq_visdate_preview = pd.to_datetime(
        faq_raw["VISDATE"],
        errors="coerce"
    )

    print("\nVisit-date coverage:")
    print(f"  Earliest valid date: {faq_visdate_preview.min()}")
    print(f"  Latest valid date:   {faq_visdate_preview.max()}")
    print(
        "  Unparseable or missing dates: "
        f"{faq_visdate_preview.isna().sum():,}"
    )
else:
    print("\nVISDATE was not found in the FAQ table.")

# Confirm that the raw dataframe remains the complete loaded source table.
print("\nRaw dataframe retained as: faq_raw")

## 1.4. Loading and inspecting the ADNI data dictionary

I am loading the ADNI data dictionary into an untouched dataframe named `datadic_raw`. first inspect its structure and identify which dictionary fields contain table names, variable names, descriptions, coding information, units, and data types.

then search for dictionary entries matching the columns in the FAQ source table. This allows me to interpret the FAQ variables from the official ADNI metadata rather than relying on assumptions.

In [ ]:
# Load the ADNI data dictionary without modifying the source CSV.
datadic_raw = pd.read_csv(datadic_path, low_memory=False)

print("Data dictionary dimensions:")
print(f"  Rows:    {datadic_raw.shape[0]:,}")
print(f"  Columns: {datadic_raw.shape[1]:,}")

print("\nData dictionary columns:")
for position, column in enumerate(datadic_raw.columns, start=1):
    print(f"{position:>3}. {column}")

print("\nSample data dictionary records:")
display(datadic_raw.head())

print("\nData dictionary data types:")
display(
    datadic_raw.dtypes
    .rename("dtype")
    .to_frame()
)

# Identify the dictionary fields expected to describe ADNI variables.
expected_dictionary_fields = [
    "FLDNAME",
    "TBLNAME",
    "CRFNAME",
    "TEXT",
    "CODE",
    "UNITS",
    "TYPE",
]

available_dictionary_fields = [
    column
    for column in expected_dictionary_fields
    if column in datadic_raw.columns
]

missing_dictionary_fields = [
    column
    for column in expected_dictionary_fields
    if column not in datadic_raw.columns
]

print("\nExpected dictionary fields found:")
print(available_dictionary_fields)

print("\nExpected dictionary fields not found:")
print(missing_dictionary_fields)

# Match FAQ source columns to dictionary rows using FLDNAME.
if "FLDNAME" in datadic_raw.columns:
    faq_column_names = set(faq_raw.columns.astype(str))

    faq_dictionary_matches = datadic_raw[
        datadic_raw["FLDNAME"]
        .astype("string")
        .str.strip()
        .isin(faq_column_names)
    ].copy()

    matched_faq_columns = set(
        faq_dictionary_matches["FLDNAME"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    unmatched_faq_columns = [
        column
        for column in faq_raw.columns
        if column not in matched_faq_columns
    ]

    print("\nFAQ-to-dictionary matching:")
    print(
        f"  FAQ columns with at least one dictionary match: "
        f"{len(matched_faq_columns):,} of {faq_raw.shape[1]:,}"
    )
    print(
        f"  FAQ columns without a dictionary match: "
        f"{len(unmatched_faq_columns):,}"
    )

    print("\nUnmatched FAQ columns:")
    for column in unmatched_faq_columns:
        print(f"  - {column}")

    # Display all matching dictionary records using the available fields.
    display_fields = [
        column
        for column in expected_dictionary_fields
        if column in faq_dictionary_matches.columns
    ]

    print("\nDictionary records matching FAQ columns:")
    display(
        faq_dictionary_matches[display_fields]
        .sort_values(
            by=[
                column
                for column in ["FLDNAME", "TBLNAME", "CRFNAME"]
                if column in display_fields
            ],
            kind="stable"
        )
        .reset_index(drop=True)
    )
else:
    print(
        "\nFLDNAME is not present in the dictionary, so direct FAQ "
        "column matching cannot yet be performed."
    )

## 1.5. Isolating FAQ-specific data dictionary records

I am narrowing the ADNI data dictionary to rows that belong specifically to the Functional Activities Questionnaire.

The previous match used only variable names, so common fields such as `RID`, `VISDATE`, and `PHASE` matched many unrelated ADNI tables. inspect table names and form names containing terms such as `FAQ`, `Functional Activities`, or `Activities Questionnaire`.

This step will identify the correct FAQ table definitions before I interpret item scores, total scores, source variables, language fields, or QC fields.

In [ ]:
# Search the dictionary table and form names for FAQ-related records.
faq_dictionary_mask = (
    datadic_raw["TBLNAME"]
    .astype("string")
    .str.contains(
        r"\bFAQ\b|FUNCTIONAL\s+ACTIVIT|ACTIVITIES\s+QUESTIONNAIRE",
        case=False,
        na=False,
        regex=True,
    )
    |
    datadic_raw["CRFNAME"]
    .astype("string")
    .str.contains(
        r"\bFAQ\b|FUNCTIONAL\s+ACTIVIT|ACTIVITIES\s+QUESTIONNAIRE",
        case=False,
        na=False,
        regex=True,
    )
)

faq_datadic_candidates = datadic_raw.loc[faq_dictionary_mask].copy()

print("FAQ-related dictionary candidate rows:")
print(f"  Rows: {len(faq_datadic_candidates):,}")

print("\nFAQ-related table names:")
display(
    faq_datadic_candidates[
        ["PHASE", "TBLNAME", "CRFNAME"]
    ]
    .drop_duplicates()
    .sort_values(
        ["TBLNAME", "CRFNAME", "PHASE"],
        kind="stable"
    )
    .reset_index(drop=True)
)

# Retain only FAQ-source columns found in the FAQ-specific dictionary rows.
faq_specific_matches = faq_datadic_candidates[
    faq_datadic_candidates["FLDNAME"]
    .astype("string")
    .str.strip()
    .isin(faq_raw.columns)
].copy()

faq_specific_matched_columns = set(
    faq_specific_matches["FLDNAME"]
    .dropna()
    .astype(str)
    .str.strip()
)

faq_specific_unmatched_columns = [
    column
    for column in faq_raw.columns
    if column not in faq_specific_matched_columns
]

print("\nFAQ source-column matching after FAQ-table filtering:")
print(
    "  Columns matched to FAQ-specific dictionary rows: "
    f"{len(faq_specific_matched_columns):,} of {faq_raw.shape[1]:,}"
)

print(
    "  Columns not matched to FAQ-specific dictionary rows: "
    f"{len(faq_specific_unmatched_columns):,}"
)

print("\nColumns without an FAQ-specific dictionary match:")
for column in faq_specific_unmatched_columns:
    print(f"  - {column}")

# Display the FAQ-specific definitions in a readable order.
faq_dictionary_display_columns = [
    "PHASE",
    "TBLNAME",
    "CRFNAME",
    "FLDNAME",
    "TEXT",
    "TYPE",
    "LENGTH",
    "CODE",
    "UNITS",
    "STATUS",
    "CODE_CHANGES",
    "MAPPING_NOTES",
]

faq_dictionary_display_columns = [
    column
    for column in faq_dictionary_display_columns
    if column in faq_specific_matches.columns
]

print("\nFAQ-specific dictionary definitions:")
display(
    faq_specific_matches[faq_dictionary_display_columns]
    .sort_values(
        ["FLDNAME", "PHASE", "TBLNAME"],
        kind="stable"
    )
    .reset_index(drop=True)
)

## 1.6. Creating an FAQ-specific data dictionary

I am creating a smaller data dictionary containing only records for the official ADNI Functional Assessment Questionnaire table.

The full ADNI dictionary contains definitions for many unrelated tables, and common variables such as `RID` and `VISDATE` appear repeatedly. Restricting the dictionary to `TBLNAME == "FAQ"` ensures that the definitions The notebook inspects belong specifically to the FAQ source file.

This step does not modify the FAQ data or make any standardisation decisions.

In [ ]:
# Select only records belonging to the official ADNI FAQ table.
faq_datadic = datadic_raw[
    datadic_raw["TBLNAME"]
    .astype("string")
    .str.strip()
    .eq("FAQ")
].copy()

# Clean surrounding whitespace in the main dictionary fields.
faq_dictionary_text_columns = [
    "PHASE",
    "CRFNAME",
    "TBLNAME",
    "FLDNAME",
    "TEXT",
    "TYPE",
    "CODE",
    "UNITS",
    "STATUS",
    "CODE_CHANGES",
    "MAPPING_NOTES",
]

for column in faq_dictionary_text_columns:
    if column in faq_datadic.columns:
        faq_datadic[column] = (
            faq_datadic[column]
            .astype("string")
            .str.strip()
        )

# Keep only dictionary variables that occur in the loaded FAQ source table.
faq_datadic = faq_datadic[
    faq_datadic["FLDNAME"].isin(faq_raw.columns)
].copy()

# Arrange the dictionary in source-column and phase order.
faq_source_column_order = {
    column: position
    for position, column in enumerate(faq_raw.columns)
}

faq_phase_order = {
    "ADNI1": 1,
    "ADNIGO": 2,
    "ADNI2": 3,
    "ADNI3": 4,
    "ADNI4": 5,
}

faq_datadic["source_column_order"] = (
    faq_datadic["FLDNAME"].map(faq_source_column_order)
)

faq_datadic["phase_order"] = (
    faq_datadic["PHASE"].map(faq_phase_order)
)

faq_datadic = (
    faq_datadic
    .sort_values(
        ["source_column_order", "phase_order"],
        kind="stable"
    )
    .drop(columns=["source_column_order", "phase_order"])
    .reset_index(drop=True)
)

print("FAQ-specific data dictionary:")
print(f"  Rows: {len(faq_datadic):,}")
print(
    "  Source variables represented: "
    f"{faq_datadic['FLDNAME'].nunique():,} of {faq_raw.shape[1]:,}"
)

faq_dictionary_unmatched_columns = [
    column
    for column in faq_raw.columns
    if column not in set(faq_datadic["FLDNAME"].dropna())
]

print("\nSource columns without an FAQ-specific dictionary definition:")
for column in faq_dictionary_unmatched_columns:
    print(f"  - {column}")

print("\nFAQ table and form names:")
display(
    faq_datadic[
        ["PHASE", "TBLNAME", "CRFNAME"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 1.7. Listing the FAQ variables and their phase coverage

I am creating one clear row per FAQ source variable.

For each variable, show:

- its official meaning;
- the ADNI phases in which it is defined;
- its dictionary data type;
- whether its description, coding, or type differs across phases.

At this stage, a difference only means the dictionary entries are not identical. It does not yet mean that recoding is required.

In [ ]:
# Preserve the original source-column order.
faq_source_column_order = {
    column: position
    for position, column in enumerate(faq_raw.columns)
}

# Define the chronological phase order.
faq_phase_order = ["ADNI1", "ADNIGO", "ADNI2", "ADNI3", "ADNI4"]


def combine_unique_values(series):
    """Combine distinct non-missing values while preserving their order."""
    values = []

    for value in series.dropna():
        value = str(value).strip()

        if value and value not in values:
            values.append(value)

    return " | ".join(values) if values else pd.NA


def combine_phases(series):
    """Return phase names in chronological order."""
    observed_phases = set(series.dropna().astype(str))

    return ", ".join(
        phase
        for phase in faq_phase_order
        if phase in observed_phases
    )


# Create one summary row per FAQ variable.
faq_variable_review = (
    faq_datadic
    .groupby("FLDNAME", sort=False)
    .agg(
        official_meaning=("TEXT", combine_unique_values),
        phases_defined=("PHASE", combine_phases),
        dictionary_types=("TYPE", combine_unique_values),
        distinct_descriptions=("TEXT", "nunique"),
        distinct_codes=("CODE", "nunique"),
        distinct_types=("TYPE", "nunique"),
    )
    .reset_index()
)

# Add variables that are present in the source file but absent from the FAQ dictionary.
unmatched_rows = pd.DataFrame(
    {
        "FLDNAME": ["PHASE", "update_stamp"],
        "official_meaning": [
            "ADNI study phase supplied by the harmonised export",
            "Database update timestamp supplied by the export",
        ],
        "phases_defined": [pd.NA, pd.NA],
        "dictionary_types": [pd.NA, pd.NA],
        "distinct_descriptions": [0, 0],
        "distinct_codes": [0, 0],
        "distinct_types": [0, 0],
    }
)

faq_variable_review = pd.concat(
    [faq_variable_review, unmatched_rows],
    ignore_index=True
)

# Convert the difference counts into readable indicators.
faq_variable_review["description_differs"] = (
    faq_variable_review["distinct_descriptions"] > 1
)

faq_variable_review["coding_differs"] = (
    faq_variable_review["distinct_codes"] > 1
)

faq_variable_review["type_differs"] = (
    faq_variable_review["distinct_types"] > 1
)

# Restore the original source-table column order.
faq_variable_review["source_column_order"] = (
    faq_variable_review["FLDNAME"].map(faq_source_column_order)
)

faq_variable_review = (
    faq_variable_review
    .sort_values("source_column_order", kind="stable")
    .drop(
        columns=[
            "source_column_order",
            "distinct_descriptions",
            "distinct_codes",
            "distinct_types",
        ]
    )
    .reset_index(drop=True)
)

print("FAQ variables and phase coverage:")
display(faq_variable_review)

## 1.8. Inspecting the information-source coding across phases

I am examining the exact coding of `SOURCE`, which identifies who provided the FAQ information.

The variable is not defined for ADNI1 but appears from ADNIGO onward. I need to determine whether its numeric values have the same meaning across phases or whether a harmonised source variable must be created.

This step only reviews the official dictionary definitions and does not modify the FAQ data.

In [ ]:
# Select the phase-specific dictionary definitions for SOURCE.
source_dictionary_review = (
    faq_datadic[
        faq_datadic["FLDNAME"].eq("SOURCE")
    ][
        [
            "PHASE",
            "TEXT",
            "TYPE",
            "CODE",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Official SOURCE definitions by ADNI phase:")

for _, row in source_dictionary_review.iterrows():
    print("\n" + "=" * 80)
    print(f"PHASE: {row['PHASE']}")
    print("=" * 80)
    print(f"Meaning: {row['TEXT']}")
    print(f"Type: {row['TYPE']}")
    print(f"Code: {row['CODE']}")
    print(f"Code changes: {row['CODE_CHANGES']}")
    print(f"Mapping notes: {row['MAPPING_NOTES']}")

## 1.9. Inspecting the FAQ item scoring across phases

I am reviewing the exact response codes used for the ten FAQ item variables.

These items should represent the same functional scoring system, but the dictionary previously showed multiple coding strings. I need to determine whether this reflects a genuine scoring difference or only formatting differences in the metadata.

This step does not alter the FAQ data.

In [ ]:
faq_item_columns = [
    "FAQFINAN",
    "FAQFORM",
    "FAQSHOP",
    "FAQGAME",
    "FAQBEVG",
    "FAQMEAL",
    "FAQEVENT",
    "FAQTV",
    "FAQREM",
    "FAQTRAVL",
]

faq_item_dictionary_review = (
    faq_datadic[
        faq_datadic["FLDNAME"].isin(faq_item_columns)
    ][
        [
            "FLDNAME",
            "PHASE",
            "TEXT",
            "CODE",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["FLDNAME", "PHASE"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("Official FAQ item coding by variable and phase:")

for variable in faq_item_columns:
    variable_rows = faq_item_dictionary_review[
        faq_item_dictionary_review["FLDNAME"].eq(variable)
    ]

    print("\n" + "=" * 100)
    print(variable)
    print("=" * 100)

    for _, row in variable_rows.iterrows():
        print(f"\nPhase: {row['PHASE']}")
        print(f"Meaning: {row['TEXT']}")
        print(f"Code: {row['CODE']}")
        print(f"Code changes: {row['CODE_CHANGES']}")
        print(f"Mapping notes: {row['MAPPING_NOTES']}")

## 1.10. FAQ item coding interpretation

The ten FAQ items use the same response and scoring system across ADNI1, ADNIGO, ADNI2, ADNI3, and ADNI4. The apparent phase-specific coding differences in the data dictionary are only formatting differences, mainly additional spaces in the ADNI4 definitions. No phase-specific recoding is required.

Each raw FAQ item can contain one of six response values:

| Raw value | Response meaning | Contribution to FAQ score |
|---:|---|---:|
| 0 | Normal | 0 |
| 1 | Never performed the activity, but could do it now | 0 |
| 2 | Never performed the activity and would now have difficulty | 1 |
| 3 | Has difficulty but completes the activity independently | 1 |
| 4 | Requires assistance | 2 |
| 5 | Dependent | 3 |

The raw item values are therefore not identical to their scored contributions. Values `0` and `1` both contribute `0`; values `2` and `3` both contribute `1`; value `4` contributes `2`; and value `5` contributes `3`.

The same mapping applies to:

- `FAQFINAN`: writing checks, paying bills, or balancing a checkbook;
- `FAQFORM`: assembling tax records, business affairs, or other papers;
- `FAQSHOP`: shopping independently;
- `FAQGAME`: playing games of skill or participating in hobbies;
- `FAQBEVG`: preparing a hot drink and safely using the stove;
- `FAQMEAL`: preparing a balanced meal;
- `FAQEVENT`: keeping track of current events;
- `FAQTV`: understanding television programmes, books, or magazines;
- `FAQREM`: remembering appointments, occasions, holidays, and medications;
- `FAQTRAVL`: travelling outside the neighbourhood, driving, or arranging transport.

The wording difference for `FAQBEVG` in some older phases is only a spelling error in the dictionary (`turing` instead of `turning`) and does not represent a change in the variable definition.

For preprocessing, the original item responses should be preserved. A separate scored version may later be created using the official mapping:

```text
0 -> 0
1 -> 0
2 -> 1
3 -> 1
4 -> 2
5 -> 3

## 1.11. Inspecting the official FAQ total-score definition

I am now reviewing the phase-specific definition of `FAQTOTAL`.

The ten FAQ items use a common scored contribution from `0` to `3`, so a complete FAQ total would normally range from `0` to `30`. However, I need to confirm from the ADNI dictionary whether the total-score definition is consistent across phases and whether any phase-specific notes describe missing items, prorating, or special calculation rules.

This step only inspects the official metadata. It does not recalculate or overwrite the reported FAQ total.

In [ ]:
# Select the phase-specific dictionary definitions for FAQTOTAL.
faq_total_dictionary_review = (
    faq_datadic[
        faq_datadic["FLDNAME"].eq("FAQTOTAL")
    ][
        [
            "PHASE",
            "TEXT",
            "TYPE",
            "LENGTH",
            "CODE",
            "UNITS",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Official FAQTOTAL definitions by ADNI phase:")

for _, row in faq_total_dictionary_review.iterrows():
    print("\n" + "=" * 80)
    print(f"PHASE: {row['PHASE']}")
    print("=" * 80)
    print(f"Meaning: {row['TEXT']}")
    print(f"Type: {row['TYPE']}")
    print(f"Length: {row['LENGTH']}")
    print(f"Code: {row['CODE']}")
    print(f"Units: {row['UNITS']}")
    print(f"Status: {row['STATUS']}")
    print(f"Code changes: {row['CODE_CHANGES']}")
    print(f"Mapping notes: {row['MAPPING_NOTES']}")

## 1.12. FAQ total-score interpretation

The official `FAQTOTAL` definition is consistent across all ADNI phases, although the data dictionary stores the validation rule differently.

For ADNI1, ADNIGO, ADNI2, and ADNI3, the permitted range is recorded in the `CODE` field as:

```text
0..30

## 1.13. Inspecting FAQ language and quality-control fields

I am now reviewing the fields that may affect data quality or interpretation rather than the FAQ score itself.

In this file, `LANGUAGE_CODE`, `HAS_QC_ERROR`, and `DD_CRF_VERSION_LABEL` are available only in ADNI4. Their absence in earlier phases is therefore structural missingness and must not be treated as a data-quality failure.

inspect their official coding before deciding how they should be retained or decoded.

In [ ]:
# Select the phase-specific definitions for the ADNI4 language,
# quality-control, and form-version fields.
faq_qc_dictionary_fields = [
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "DD_CRF_VERSION_LABEL",
]

faq_qc_dictionary_review = (
    faq_datadic[
        faq_datadic["FLDNAME"].isin(faq_qc_dictionary_fields)
    ][
        [
            "FLDNAME",
            "PHASE",
            "TEXT",
            "TYPE",
            "LENGTH",
            "CODE",
            "UNITS",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["FLDNAME", "PHASE"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("Official language, QC, and form-version definitions:")

for variable in faq_qc_dictionary_fields:
    variable_rows = faq_qc_dictionary_review[
        faq_qc_dictionary_review["FLDNAME"].eq(variable)
    ]

    print("\n" + "=" * 80)
    print(variable)
    print("=" * 80)

    if variable_rows.empty:
        print("No FAQ-specific dictionary definition was found.")
        continue

    for _, row in variable_rows.iterrows():
        print(f"\nPhase: {row['PHASE']}")
        print(f"Meaning: {row['TEXT']}")
        print(f"Type: {row['TYPE']}")
        print(f"Length: {row['LENGTH']}")
        print(f"Code: {row['CODE']}")
        print(f"Units: {row['UNITS']}")
        print(f"Status: {row['STATUS']}")
        print(f"Code changes: {row['CODE_CHANGES']}")
        print(f"Mapping notes: {row['MAPPING_NOTES']}")

## 1.14. Interpretation of ADNI4 language and QC fields

Three FAQ fields are defined only for ADNI4:

| Variable | Meaning | Coding | Preprocessing decision |
|---|---|---|---|
| `LANGUAGE_CODE` | Language used for the questionnaire | `e = English` | Retain for provenance and phase-specific QC. Do not use missing values in earlier phases as an exclusion criterion. |
| `HAS_QC_ERROR` | Whether the record has an unresolved quality-control error | `0 = no QC error, or the error has been approved`; `1 = has QC error` | Retain and create a separate QC flag for records with value `1`. |
| `DD_CRF_VERSION_LABEL` | Version of the ADNI4 case-report form | No coded categories documented | Retain temporarily for provenance and form-version review, but do not use as a clinical model feature. |

The absence of these variables in ADNI1, ADNIGO, ADNI2, and ADNI3 is structural missingness because they were not defined for those phases.

For `HAS_QC_ERROR`, only the value `1` should indicate a provisional QC concern. A value of `0` includes both records with no QC error and records whose QC issue was reviewed and approved, so these records should not be excluded on the basis of this field.

No cross-phase harmonisation is needed for `LANGUAGE_CODE` or `DD_CRF_VERSION_LABEL` because they are ADNI4-specific metadata. `HAS_QC_ERROR` should remain phase-specific, with earlier-phase missing values interpreted as “not collected” rather than “no error.”

## 1.15. Selecting columns for the longitudinal FAQ working table

I am now creating a working copy of the FAQ table while preserving `faq_raw` unchanged.

retain:

- participant and visit identifiers;
- all ten FAQ item responses;
- the official FAQ total;
- the information-source field;
- ADNI4 language and QC fields;
- selected provenance fields that may help with duplicate review.

exclude database timestamps and administrative update fields that do not contribute to clinical interpretation or QC.

In [ ]:
# Define the clinically relevant and QC-related FAQ columns.
faq_identifier_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "VISDATE",
]

faq_context_columns = [
    "SOURCE",
    "SPID",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "DD_CRF_VERSION_LABEL",
]

faq_item_columns = [
    "FAQFINAN",
    "FAQFORM",
    "FAQSHOP",
    "FAQGAME",
    "FAQBEVG",
    "FAQMEAL",
    "FAQEVENT",
    "FAQTV",
    "FAQREM",
    "FAQTRAVL",
]

faq_total_columns = [
    "FAQTOTAL",
]

# Retain record identifiers temporarily because they may help with duplicate review.
faq_provenance_columns = [
    "ID",
    "SITEID",
]

faq_working_columns = (
    faq_identifier_columns
    + faq_context_columns
    + faq_item_columns
    + faq_total_columns
    + faq_provenance_columns
)

# Confirm that every requested column exists before creating the working copy.
missing_working_columns = [
    column
    for column in faq_working_columns
    if column not in faq_raw.columns
]

if missing_working_columns:
    raise KeyError(
        "The following expected FAQ columns are missing: "
        + ", ".join(missing_working_columns)
    )

# Create an independent working dataframe.
faq_working = faq_raw[faq_working_columns].copy()

# Record the fields not retained in the working table.
faq_columns_not_retained = [
    column
    for column in faq_raw.columns
    if column not in faq_working_columns
]

print("FAQ working table created:")
print(f"  Rows:    {faq_working.shape[0]:,}")
print(f"  Columns: {faq_working.shape[1]:,}")

print("\nRetained columns:")
for column in faq_working.columns:
    print(f"  - {column}")

print("\nColumns not retained:")
for column in faq_columns_not_retained:
    print(f"  - {column}")

print("\nRaw dataframe remains unchanged:")
print(f"  faq_raw shape:     {faq_raw.shape}")
print(f"  faq_working shape: {faq_working.shape}")

## 1.16. Standardising FAQ data types

I am creating a typed cleaning copy named `faq_clean` from the reduced longitudinal working table.

I will:

- convert `VISDATE` to a proper pandas datetime;
- store participant, visit, phase, language, and form-version fields as string values;
- convert FAQ items, the official total, source, QC status, and numeric identifiers using `errors="coerce"`;
- preserve missing values rather than replacing them with zero;
- retain `faq_raw` and `faq_working` unchanged.

Values that cannot be converted to the expected type will become missing in `faq_clean` and will be counted for review.

In [ ]:
# Create a separate typed cleaning copy.
faq_clean = faq_working.copy()

# Preserve the original visit-date text for transparent review.
faq_clean["VISDATE_ORIGINAL"] = faq_clean["VISDATE"]

# Convert the clinical visit date to pandas datetime.
faq_clean["VISDATE"] = pd.to_datetime(
    faq_clean["VISDATE"],
    errors="coerce"
)

# Standardise text-based identifiers and contextual fields.
faq_string_columns = [
    "PHASE",
    "PTID",
    "VISCODE",
    "VISCODE2",
    "LANGUAGE_CODE",
    "DD_CRF_VERSION_LABEL",
]

for column in faq_string_columns:
    faq_clean[column] = (
        faq_clean[column]
        .astype("string")
        .str.strip()
    )

# Convert numeric identifiers while preserving missing values.
faq_numeric_identifier_columns = [
    "RID",
    "SPID",
    "ID",
    "SITEID",
]

for column in faq_numeric_identifier_columns:
    faq_clean[column] = pd.to_numeric(
        faq_clean[column],
        errors="coerce"
    ).astype("Int64")

# Convert source and QC fields to nullable integer values.
faq_coded_context_columns = [
    "SOURCE",
    "HAS_QC_ERROR",
]

for column in faq_coded_context_columns:
    faq_clean[column] = pd.to_numeric(
        faq_clean[column],
        errors="coerce"
    ).astype("Int64")

# Convert the ten raw FAQ item responses and reported total to numeric.
faq_score_columns = faq_item_columns + ["FAQTOTAL"]

for column in faq_score_columns:
    faq_clean[column] = pd.to_numeric(
        faq_clean[column],
        errors="coerce"
    )

# Compare missingness before and after conversion to identify parsing losses.
conversion_review_rows = []

for column in (
    ["VISDATE"]
    + faq_numeric_identifier_columns
    + faq_coded_context_columns
    + faq_score_columns
):
    original_column = (
        "VISDATE_ORIGINAL"
        if column == "VISDATE"
        else column
    )

    original_missing = faq_working[
        "VISDATE" if column == "VISDATE" else column
    ].isna().sum()

    converted_missing = faq_clean[column].isna().sum()

    conversion_review_rows.append(
        {
            "column": column,
            "original_missing": int(original_missing),
            "missing_after_conversion": int(converted_missing),
            "new_missing_from_conversion": int(
                converted_missing - original_missing
            ),
        }
    )

faq_type_conversion_review = pd.DataFrame(conversion_review_rows)

print("Typed FAQ cleaning copy created:")
print(f"  Rows:    {faq_clean.shape[0]:,}")
print(f"  Columns: {faq_clean.shape[1]:,}")

print("\nData types after standardisation:")
display(
    faq_clean.dtypes
    .rename("dtype")
    .to_frame()
)

print("\nConversion review:")
display(faq_type_conversion_review)

print("\nVisit-date coverage after conversion:")
print(f"  Earliest valid date: {faq_clean['VISDATE'].min()}")
print(f"  Latest valid date:   {faq_clean['VISDATE'].max()}")
print(
    "  Missing or unparseable dates: "
    f"{faq_clean['VISDATE'].isna().sum():,}"
)

## 1.17. Reviewing missingness by ADNI phase

I am now examining how often each FAQ field is missing within each ADNI phase.

This is necessary because several variables were introduced only in later phases. Missing values in fields such as `SPID`, `LANGUAGE_CODE`, `HAS_QC_ERROR`, and `DD_CRF_VERSION_LABEL` may therefore represent structural missingness rather than incomplete records.

For the ten FAQ items and `FAQTOTAL`, which are defined across all phases, missingness is more likely to represent record-level incompleteness and will require closer review.

This step only summarises missingness. It does not exclude any records.

In [ ]:
# Define the fields to review by phase.
faq_phase_missingness_columns = [
    "SOURCE",
    "SPID",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "DD_CRF_VERSION_LABEL",
    *faq_item_columns,
    "FAQTOTAL",
]

# Count the number of records in each phase.
phase_row_counts = (
    faq_clean
    .groupby("PHASE", dropna=False)
    .size()
    .rename("phase_row_count")
)

# Calculate missing counts by phase.
faq_missing_count_by_phase = (
    faq_clean
    .groupby("PHASE", dropna=False)[faq_phase_missingness_columns]
    .agg(lambda column: column.isna().sum())
)

# Convert missing counts to percentages within each phase.
faq_missing_percentage_by_phase = (
    faq_missing_count_by_phase
    .div(phase_row_counts, axis=0)
    .mul(100)
    .round(2)
)

print("Rows in each ADNI phase:")
display(
    phase_row_counts
    .reset_index()
)

print("\nMissing-value counts by phase:")
display(faq_missing_count_by_phase)

print("\nMissing-value percentages by phase:")
display(faq_missing_percentage_by_phase)

# Create a long-format summary that is easier to filter and save later.
faq_phase_missingness_summary = (
    faq_missing_count_by_phase
    .stack()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"level_1": "variable"})
)

faq_phase_missingness_summary["phase_row_count"] = (
    faq_phase_missingness_summary["PHASE"]
    .map(phase_row_counts)
)

faq_phase_missingness_summary["missing_percentage"] = (
    faq_phase_missingness_summary["missing_count"]
    / faq_phase_missingness_summary["phase_row_count"]
    * 100
).round(2)

print("\nLong-format phase missingness summary:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(faq_phase_missingness_summary)

## 1.18. Inspecting observed information-source values by phase

The FAQ data dictionary defines `SOURCE` from ADNIGO onward, but the loaded FAQ table also contains non-missing `SOURCE` values for every ADNI1 record.

I am therefore inspecting the actual values and frequencies of `SOURCE` within each phase. This will show whether ADNI1 uses the same `1` and `2` coding as later phases or contains additional values that require separate interpretation.

not recode or exclude any records in this step.

In [ ]:
# Count observed SOURCE values separately within each ADNI phase.
faq_source_distribution = (
    faq_clean
    .assign(
        SOURCE_DISPLAY=faq_clean["SOURCE"]
        .astype("string")
        .fillna("<MISSING>")
    )
    .groupby(
        ["PHASE", "SOURCE_DISPLAY"],
        dropna=False
    )
    .size()
    .rename("row_count")
    .reset_index()
)

# Add percentages calculated within each phase.
faq_source_distribution["phase_row_count"] = (
    faq_source_distribution["PHASE"]
    .map(
        faq_clean
        .groupby("PHASE")
        .size()
    )
)

faq_source_distribution["percentage_within_phase"] = (
    faq_source_distribution["row_count"]
    / faq_source_distribution["phase_row_count"]
    * 100
).round(2)

# Apply a provisional readable label only to known documented codes.
source_label_map = {
    "1": "Participant visit / in-person",
    "2": "Telephone or video call / remote",
    "<MISSING>": "Missing",
}

faq_source_distribution["documented_interpretation"] = (
    faq_source_distribution["SOURCE_DISPLAY"]
    .map(source_label_map)
    .fillna("Undocumented value requiring review")
)

print("Observed SOURCE values by phase:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(
        faq_source_distribution.sort_values(
            ["PHASE", "SOURCE_DISPLAY"],
            kind="stable"
        )
    )

# List all distinct non-missing values found in the source table.
print("\nDistinct non-missing SOURCE values in the complete FAQ table:")
print(
    sorted(
        faq_clean["SOURCE"]
        .dropna()
        .unique()
        .tolist()
    )
)

## 1.19. Investigating the undocumented ADNI1 SOURCE value

I found 23 ADNI1 records with `SOURCE = -1`, while all other non-missing source values are `1` or `2`.

Because `-1` is not defined in the FAQ-specific dictionary, I will:

- inspect the affected records;
- check their visit codes, dates, scores, and totals;
- search the full ADNI dictionary for any definition or mapping note involving `SOURCE = -1`;
- determine whether `-1` is a special missing code, an administrative placeholder, or a valid historical value.

not recode these records until their meaning is supported by the available metadata.

In [ ]:
# Select all FAQ records with the undocumented SOURCE value.
faq_source_minus_one = faq_clean[
    faq_clean["SOURCE"].eq(-1)
].copy()

print("Records with SOURCE = -1:")
print(f"  Rows: {len(faq_source_minus_one):,}")
print(
    "  Unique participants: "
    f"{faq_source_minus_one['RID'].nunique(dropna=True):,}"
)

# Show their phase, visits, dates, items, and totals.
source_minus_one_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "VISDATE",
    "SOURCE",
    *faq_item_columns,
    "FAQTOTAL",
    "ID",
    "SITEID",
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(
        faq_source_minus_one[source_minus_one_columns]
        .sort_values(
            ["RID", "VISDATE", "VISCODE2"],
            kind="stable"
        )
        .reset_index(drop=True)
    )

# Summarise the visit-code distribution for these records.
print("\nVisit-code distribution for SOURCE = -1:")

source_minus_one_visit_summary = (
    faq_source_minus_one
    .groupby(
        ["VISCODE", "VISCODE2"],
        dropna=False
    )
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values(
        "row_count",
        ascending=False,
        kind="stable"
    )
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(source_minus_one_visit_summary)

# Search the full ADNI dictionary for references to SOURCE and -1.
source_dictionary_all_tables = datadic_raw[
    datadic_raw["FLDNAME"]
    .astype("string")
    .str.strip()
    .eq("SOURCE")
].copy()

source_minus_one_dictionary_matches = source_dictionary_all_tables[
    source_dictionary_all_tables[
        [
            "CODE",
            "CODE_CHANGES",
            "MAPPING_NOTES",
            "TEXT",
        ]
    ]
    .astype("string")
    .apply(
        lambda column: column.str.contains(
            r"(^|[^0-9])-1([^0-9]|$)",
            regex=True,
            na=False
        )
    )
    .any(axis=1)
].copy()

print("\nDictionary rows mentioning SOURCE = -1:")
print(f"  Matching rows: {len(source_minus_one_dictionary_matches):,}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(
        source_minus_one_dictionary_matches[
            [
                "PHASE",
                "TBLNAME",
                "CRFNAME",
                "FLDNAME",
                "TEXT",
                "CODE",
                "CODE_CHANGES",
                "MAPPING_NOTES",
            ]
        ]
        .reset_index(drop=True)
    )

## 1.20. Inspecting observed FAQ score values

I am reviewing every distinct value found in the ten FAQ item fields and in `FAQTOTAL`.

The official valid ranges are:

- raw FAQ item responses: `0` to `5`;
- reported FAQ total: `0` to `30`.

I have already found 23 ADNI1 records in which `SOURCE`, every FAQ item, and `FAQTOTAL` are all recorded as `-1`. This strongly suggests an undocumented missing-record placeholder, but first determine whether `-1` or any other unexpected values occur elsewhere.

This step only inspects the observed values. It does not replace or remove anything.

In [ ]:
# Define the official valid ranges.
faq_item_valid_values = {0, 1, 2, 3, 4, 5}
faq_total_min = 0
faq_total_max = 30

# Build a complete value-frequency table for all FAQ items and the total.
faq_value_distribution_rows = []

for column in faq_item_columns + ["FAQTOTAL"]:
    value_counts = (
        faq_clean[column]
        .value_counts(dropna=False)
        .sort_index()
    )

    for value, count in value_counts.items():
        if pd.isna(value):
            value_display = "<MISSING>"
            value_status = "missing"

        elif column in faq_item_columns:
            value_display = value

            if value in faq_item_valid_values:
                value_status = "valid item response"
            else:
                value_status = "outside official item range"

        else:
            value_display = value

            if (
                faq_total_min <= value <= faq_total_max
                and float(value).is_integer()
            ):
                value_status = "valid total"
            else:
                value_status = "outside official total definition"

        faq_value_distribution_rows.append(
            {
                "variable": column,
                "value": value_display,
                "row_count": int(count),
                "value_status": value_status,
            }
        )

faq_value_distribution = pd.DataFrame(
    faq_value_distribution_rows
)

print("Observed FAQ values and frequencies:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(faq_value_distribution)

# Isolate all non-missing values outside the official definitions.
faq_unexpected_values = faq_value_distribution[
    faq_value_distribution["value_status"].isin(
        [
            "outside official item range",
            "outside official total definition",
        ]
    )
].copy()

print("\nUnexpected FAQ values:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(faq_unexpected_values)

# Identify rows containing at least one out-of-range item or total value.
item_out_of_range_mask = (
    faq_clean[faq_item_columns]
    .notna()
    & ~faq_clean[faq_item_columns].isin(faq_item_valid_values)
).any(axis=1)

total_out_of_range_mask = (
    faq_clean["FAQTOTAL"].notna()
    & (
        (faq_clean["FAQTOTAL"] < faq_total_min)
        | (faq_clean["FAQTOTAL"] > faq_total_max)
        | (faq_clean["FAQTOTAL"] % 1 != 0)
    )
)

print("\nRows with at least one item outside 0–5:")
print(f"  {item_out_of_range_mask.sum():,}")

print("\nRows with FAQTOTAL outside integer range 0–30:")
print(f"  {total_out_of_range_mask.sum():,}")

print("\nRows where every FAQ item and FAQTOTAL equal -1:")
all_faq_minus_one_mask = (
    faq_clean[faq_item_columns + ["FAQTOTAL"]]
    .eq(-1)
    .all(axis=1)
)

print(f"  {all_faq_minus_one_mask.sum():,}")

## 1.21. Inspecting mixed negative-one FAQ records

I found 32 records containing the undocumented value `-1`.

Most of them have `-1` for every FAQ item and for `FAQTOTAL`, which strongly suggests a completely unavailable questionnaire. However, six records contain a mixture of `-1`, valid item responses, and possibly missing values.

I am inspecting those mixed records separately before deciding how to recode them. This is necessary because a partially completed questionnaire should not automatically be treated the same as a completely unavailable questionnaire.

This step does not modify or exclude any records.

In [ ]:
faq_score_review_columns = faq_item_columns + ["FAQTOTAL"]

# Identify all rows containing at least one -1 value.
faq_any_minus_one_mask = (
    faq_clean[faq_score_review_columns]
    .eq(-1)
    .any(axis=1)
)

# Identify rows where all ten items and the total are -1.
faq_all_minus_one_mask = (
    faq_clean[faq_score_review_columns]
    .eq(-1)
    .all(axis=1)
)

# Mixed records contain at least one -1 but are not entirely -1.
faq_mixed_minus_one_mask = (
    faq_any_minus_one_mask
    & ~faq_all_minus_one_mask
)

faq_mixed_minus_one_records = (
    faq_clean.loc[
        faq_mixed_minus_one_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "SOURCE",
            *faq_item_columns,
            "FAQTOTAL",
            "HAS_QC_ERROR",
            "ID",
            "SITEID",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("Negative-one record patterns:")
print(
    "  Rows with at least one -1: "
    f"{faq_any_minus_one_mask.sum():,}"
)
print(
    "  Rows where all items and total are -1: "
    f"{faq_all_minus_one_mask.sum():,}"
)
print(
    "  Rows with a mixed -1 pattern: "
    f"{faq_mixed_minus_one_mask.sum():,}"
)

print("\nMixed -1 records:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(faq_mixed_minus_one_records)

# Summarise the pattern of -1, missing, and valid values in each mixed record.
faq_mixed_pattern_summary = (
    faq_clean.loc[
        faq_mixed_minus_one_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            *faq_score_review_columns,
        ]
    ]
    .copy()
)

faq_mixed_pattern_summary["minus_one_item_count"] = (
    faq_mixed_pattern_summary[faq_item_columns]
    .eq(-1)
    .sum(axis=1)
)

faq_mixed_pattern_summary["missing_item_count"] = (
    faq_mixed_pattern_summary[faq_item_columns]
    .isna()
    .sum(axis=1)
)

faq_mixed_pattern_summary["valid_item_count"] = (
    faq_mixed_pattern_summary[faq_item_columns]
    .isin({0, 1, 2, 3, 4, 5})
    .sum(axis=1)
)

faq_mixed_pattern_summary["total_is_minus_one"] = (
    faq_mixed_pattern_summary["FAQTOTAL"].eq(-1)
)

print("\nMixed-record pattern summary:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        faq_mixed_pattern_summary[
            [
                "PHASE",
                "PTID",
                "RID",
                "VISCODE2",
                "minus_one_item_count",
                "missing_item_count",
                "valid_item_count",
                "FAQTOTAL",
                "total_is_minus_one",
            ]
        ].reset_index(drop=True)
    )

## 1.22. Converting undocumented negative-one codes to missing values

I found that `-1` is used as an undocumented unavailable-value code in the FAQ data.

There are two distinct patterns:

- records where all ten FAQ items and `FAQTOTAL` are `-1`, indicating a fully unavailable questionnaire;
- records where only some items are `-1` and the remaining items are valid, indicating a partially unavailable questionnaire.

preserve the original values for traceability, create separate QC flags for the two patterns, and convert `-1` to missing in the analytical FAQ fields.

not remove these records or reconstruct the total score at this stage.

In [ ]:
# Preserve the original FAQ item and total values before recoding.
for column in faq_item_columns + ["FAQTOTAL"]:
    original_column = f"{column}_ORIGINAL"

    if original_column not in faq_clean.columns:
        faq_clean[original_column] = faq_clean[column]

# Identify records containing undocumented -1 values.
faq_any_minus_one_mask = (
    faq_clean[faq_item_columns + ["FAQTOTAL"]]
    .eq(-1)
    .any(axis=1)
)

faq_all_minus_one_mask = (
    faq_clean[faq_item_columns + ["FAQTOTAL"]]
    .eq(-1)
    .all(axis=1)
)

faq_partial_minus_one_mask = (
    faq_any_minus_one_mask
    & ~faq_all_minus_one_mask
)

# Create transparent QC flags before replacing any values.
faq_clean["flag_faq_fully_unavailable_minus_one"] = (
    faq_all_minus_one_mask
)

faq_clean["flag_faq_partially_unavailable_minus_one"] = (
    faq_partial_minus_one_mask
)

faq_clean["flag_any_faq_minus_one"] = (
    faq_any_minus_one_mask
)

# Count how many item-level -1 values occur in each record.
faq_clean["faq_minus_one_item_count"] = (
    faq_clean[faq_item_columns]
    .eq(-1)
    .sum(axis=1)
)

# Convert undocumented -1 values to missing in the analytical fields.
faq_clean[faq_item_columns + ["FAQTOTAL"]] = (
    faq_clean[faq_item_columns + ["FAQTOTAL"]]
    .replace(-1, pd.NA)
    .astype("Float64")
)

print("Negative-one recoding completed:")
print(
    "  Fully unavailable FAQ records: "
    f"{faq_clean['flag_faq_fully_unavailable_minus_one'].sum():,}"
)
print(
    "  Partially unavailable FAQ records: "
    f"{faq_clean['flag_faq_partially_unavailable_minus_one'].sum():,}"
)
print(
    "  Total records affected: "
    f"{faq_clean['flag_any_faq_minus_one'].sum():,}"
)

print("\nRemaining -1 values in analytical FAQ items:")
print(
    faq_clean[faq_item_columns]
    .eq(-1)
    .sum()
    .sum()
)

print("\nRemaining -1 values in analytical FAQTOTAL:")
print(
    faq_clean["FAQTOTAL"]
    .eq(-1)
    .sum()
)

print("\nMissingness after recoding:")
faq_missingness_after_minus_one_recode = (
    faq_clean[faq_item_columns + ["FAQTOTAL"]]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(faq_missingness_after_minus_one_recode)

print("\nAffected records after recoding:")

affected_faq_records = faq_clean.loc[
    faq_clean["flag_any_faq_minus_one"],
    [
        "PHASE",
        "PTID",
        "RID",
        "VISCODE2",
        "VISDATE",
        "SOURCE",
        *faq_item_columns,
        "FAQTOTAL",
        "faq_minus_one_item_count",
        "flag_faq_fully_unavailable_minus_one",
        "flag_faq_partially_unavailable_minus_one",
    ]
].copy()

affected_faq_records = affected_faq_records.sort_values(
    ["RID", "VISDATE"],
    kind="stable"
).reset_index(drop=True)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(affected_faq_records)

## 1.23. Correcting the negative-one flags from preserved original values

The previous cell was rerun after the analytical `-1` values had already been converted to missing. As a result, the negative-one flags were incorrectly recalculated from the cleaned columns.

reconstruct the flags from the preserved `*_ORIGINAL` columns, which still contain the source values. This will restore the correct distinction between:

- fully unavailable questionnaires;
- partially unavailable questionnaires;
- ordinary missing FAQ records that never contained `-1`.

The analytical FAQ fields will remain unchanged, with `-1` represented as missing.

In [ ]:
# Define the preserved original FAQ item and total columns.
faq_original_item_columns = [
    f"{column}_ORIGINAL"
    for column in faq_item_columns
]

faq_original_total_column = "FAQTOTAL_ORIGINAL"

faq_original_score_columns = (
    faq_original_item_columns
    + [faq_original_total_column]
)

# Confirm that the preserved source columns exist.
missing_original_columns = [
    column
    for column in faq_original_score_columns
    if column not in faq_clean.columns
]

if missing_original_columns:
    raise KeyError(
        "The following preserved original columns are missing: "
        + ", ".join(missing_original_columns)
    )

# Reconstruct all -1 patterns from the preserved original source values.
faq_original_any_minus_one_mask = (
    faq_clean[faq_original_score_columns]
    .eq(-1)
    .any(axis=1)
)

faq_original_all_minus_one_mask = (
    faq_clean[faq_original_score_columns]
    .eq(-1)
    .all(axis=1)
)

faq_original_partial_minus_one_mask = (
    faq_original_any_minus_one_mask
    & ~faq_original_all_minus_one_mask
)

# Replace the incorrect flags with the correctly reconstructed flags.
faq_clean["flag_faq_fully_unavailable_minus_one"] = (
    faq_original_all_minus_one_mask
)

faq_clean["flag_faq_partially_unavailable_minus_one"] = (
    faq_original_partial_minus_one_mask
)

faq_clean["flag_any_faq_minus_one"] = (
    faq_original_any_minus_one_mask
)

# Recalculate the number of original item-level -1 values per record.
faq_clean["faq_minus_one_item_count"] = (
    faq_clean[faq_original_item_columns]
    .eq(-1)
    .sum(axis=1)
)

# Confirm that analytical FAQ fields still contain no -1 values.
remaining_item_minus_one_count = (
    faq_clean[faq_item_columns]
    .eq(-1)
    .sum()
    .sum()
)

remaining_total_minus_one_count = (
    faq_clean["FAQTOTAL"]
    .eq(-1)
    .sum()
)

print("Corrected negative-one flags:")
print(
    "  Fully unavailable FAQ records: "
    f"{faq_clean['flag_faq_fully_unavailable_minus_one'].sum():,}"
)
print(
    "  Partially unavailable FAQ records: "
    f"{faq_clean['flag_faq_partially_unavailable_minus_one'].sum():,}"
)
print(
    "  Total records affected: "
    f"{faq_clean['flag_any_faq_minus_one'].sum():,}"
)

print("\nRemaining -1 values in analytical fields:")
print(f"  FAQ items: {remaining_item_minus_one_count:,}")
print(f"  FAQTOTAL:  {remaining_total_minus_one_count:,}")

# Display the correctly identified affected records.
affected_faq_records = faq_clean.loc[
    faq_clean["flag_any_faq_minus_one"],
    [
        "PHASE",
        "PTID",
        "RID",
        "VISCODE2",
        "VISDATE",
        "SOURCE",
        *faq_item_columns,
        "FAQTOTAL",
        "faq_minus_one_item_count",
        "flag_faq_fully_unavailable_minus_one",
        "flag_faq_partially_unavailable_minus_one",
    ]
].copy()

affected_faq_records = affected_faq_records.sort_values(
    ["RID", "VISDATE"],
    kind="stable"
).reset_index(drop=True)

print("\nAffected records after corrected flag reconstruction:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(affected_faq_records)

## 1.24. Validating cleaned FAQ item and total-score ranges

I am validating the analytical FAQ values after converting the undocumented `-1` codes to missing.

The official valid ranges are:

- each raw FAQ item: `0` to `5`;
- `FAQTOTAL`: integer values from `0` to `30`.

create separate flags for:

- an item outside the permitted range;
- a total outside the permitted range;
- a non-integer item or total value;
- a missing official total.

This step does not remove any records.

In [ ]:
# Check whether each non-missing FAQ item lies within the official range 0-5.
faq_item_out_of_range_matrix = (
    faq_clean[faq_item_columns].notna()
    & (
        (faq_clean[faq_item_columns] < 0)
        | (faq_clean[faq_item_columns] > 5)
    )
)

# Check for non-integer FAQ item values.
faq_item_non_integer_matrix = (
    faq_clean[faq_item_columns].notna()
    & (faq_clean[faq_item_columns] % 1 != 0)
)

# Create row-level FAQ item validation flags.
faq_clean["flag_item_out_of_range"] = (
    faq_item_out_of_range_matrix.any(axis=1)
)

faq_clean["flag_item_non_integer"] = (
    faq_item_non_integer_matrix.any(axis=1)
)

# Validate the official FAQ total.
faq_clean["flag_total_out_of_range"] = (
    faq_clean["FAQTOTAL"].notna()
    & (
        (faq_clean["FAQTOTAL"] < 0)
        | (faq_clean["FAQTOTAL"] > 30)
    )
)

faq_clean["flag_total_non_integer"] = (
    faq_clean["FAQTOTAL"].notna()
    & (faq_clean["FAQTOTAL"] % 1 != 0)
)

faq_clean["flag_missing_total"] = (
    faq_clean["FAQTOTAL"].isna()
)

# Count the number of available and missing FAQ items per record.
faq_clean["faq_available_item_count"] = (
    faq_clean[faq_item_columns]
    .notna()
    .sum(axis=1)
)

faq_clean["faq_missing_item_count"] = (
    faq_clean[faq_item_columns]
    .isna()
    .sum(axis=1)
)

# Summarise the validation flags.
faq_range_validation_summary = pd.DataFrame(
    {
        "check": [
            "Rows with at least one item outside 0–5",
            "Rows with at least one non-integer item",
            "Rows with FAQTOTAL outside 0–30",
            "Rows with non-integer FAQTOTAL",
            "Rows with missing FAQTOTAL",
            "Rows with all 10 FAQ items available",
            "Rows with at least one FAQ item missing",
            "Rows with all 10 FAQ items missing",
        ],
        "row_count": [
            int(faq_clean["flag_item_out_of_range"].sum()),
            int(faq_clean["flag_item_non_integer"].sum()),
            int(faq_clean["flag_total_out_of_range"].sum()),
            int(faq_clean["flag_total_non_integer"].sum()),
            int(faq_clean["flag_missing_total"].sum()),
            int((faq_clean["faq_available_item_count"] == 10).sum()),
            int((faq_clean["faq_missing_item_count"] > 0).sum()),
            int((faq_clean["faq_missing_item_count"] == 10).sum()),
        ],
    }
)

faq_range_validation_summary["percentage"] = (
    faq_range_validation_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("FAQ range and completeness validation:")
display(faq_range_validation_summary)

# Display any remaining range or integer violations.
faq_range_problem_mask = (
    faq_clean[
        [
            "flag_item_out_of_range",
            "flag_item_non_integer",
            "flag_total_out_of_range",
            "flag_total_non_integer",
        ]
    ]
    .any(axis=1)
)

print("\nRecords with remaining range or integer violations:")
print(f"  Rows: {faq_range_problem_mask.sum():,}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        faq_clean.loc[
            faq_range_problem_mask,
            [
                "PHASE",
                "PTID",
                "RID",
                "VISCODE2",
                "VISDATE",
                *faq_item_columns,
                "FAQTOTAL",
                "flag_item_out_of_range",
                "flag_item_non_integer",
                "flag_total_out_of_range",
                "flag_total_non_integer",
            ]
        ].reset_index(drop=True)
    )

## 1.25. Comparing the reported FAQ total with the official item-score sum

I am checking whether the reported `FAQTOTAL` agrees with the total obtained from the ten complete FAQ item responses.

The raw item responses use the official scoring conversion:

| Raw response | Scored contribution |
|---:|---:|
| 0 | 0 |
| 1 | 0 |
| 2 | 1 |
| 3 | 1 |
| 4 | 2 |
| 5 | 3 |

calculate the item-derived total only when all ten analytical FAQ items are available. not calculate or prorate a total for incomplete questionnaires.

The official `FAQTOTAL` will be preserved. Any disagreement will be recorded as a QC flag rather than automatically corrected.

In [ ]:
# Define the official conversion from raw FAQ responses to scored contributions.
faq_item_score_map = {
    0: 0,
    1: 0,
    2: 1,
    3: 1,
    4: 2,
    5: 3,
}

# Create separate scored versions of the ten FAQ items.
faq_scored_item_columns = []

for column in faq_item_columns:
    scored_column = f"{column}_SCORED"
    faq_scored_item_columns.append(scored_column)

    faq_clean[scored_column] = (
        faq_clean[column]
        .map(faq_item_score_map)
        .astype("Float64")
    )

# Identify records for which all ten analytical item responses are available.
faq_complete_items_mask = (
    faq_clean[faq_item_columns]
    .notna()
    .all(axis=1)
)

# Calculate an item-derived total only for complete questionnaires.
faq_clean["FAQTOTAL_CALCULATED"] = pd.Series(
    pd.NA,
    index=faq_clean.index,
    dtype="Float64",
)

faq_clean.loc[
    faq_complete_items_mask,
    "FAQTOTAL_CALCULATED"
] = (
    faq_clean.loc[
        faq_complete_items_mask,
        faq_scored_item_columns
    ]
    .sum(axis=1)
    .astype("Float64")
)

# Calculate the difference between the reported and item-derived totals.
faq_clean["FAQTOTAL_DIFFERENCE"] = (
    faq_clean["FAQTOTAL"]
    - faq_clean["FAQTOTAL_CALCULATED"]
)

# Flag discrepancies only when both totals are available.
faq_clean["flag_total_item_discrepancy"] = (
    faq_clean["FAQTOTAL"].notna()
    & faq_clean["FAQTOTAL_CALCULATED"].notna()
    & faq_clean["FAQTOTAL_DIFFERENCE"].ne(0)
)

# Summarise total-score agreement.
complete_with_reported_total_mask = (
    faq_clean["FAQTOTAL"].notna()
    & faq_clean["FAQTOTAL_CALCULATED"].notna()
)

faq_total_agreement_summary = pd.DataFrame(
    {
        "check": [
            "Rows with all 10 items available",
            "Complete-item rows with reported FAQTOTAL",
            "Rows where reported and calculated totals agree",
            "Rows where reported and calculated totals disagree",
            "Incomplete-item rows not assigned a calculated total",
        ],
        "row_count": [
            int(faq_complete_items_mask.sum()),
            int(complete_with_reported_total_mask.sum()),
            int(
                (
                    complete_with_reported_total_mask
                    & faq_clean["FAQTOTAL_DIFFERENCE"].eq(0)
                ).sum()
            ),
            int(faq_clean["flag_total_item_discrepancy"].sum()),
            int((~faq_complete_items_mask).sum()),
        ],
    }
)

faq_total_agreement_summary["percentage_of_all_rows"] = (
    faq_total_agreement_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("FAQ reported-total agreement summary:")
display(faq_total_agreement_summary)

# Display the distribution of any observed differences.
faq_total_difference_distribution = (
    faq_clean.loc[
        complete_with_reported_total_mask,
        "FAQTOTAL_DIFFERENCE"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("reported_minus_calculated")
    .reset_index(name="row_count")
)

print("\nDistribution of reported minus calculated FAQ totals:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(faq_total_difference_distribution)

# Display every discrepancy for transparent review.
faq_total_discrepancy_records = (
    faq_clean.loc[
        faq_clean["flag_total_item_discrepancy"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            *faq_item_columns,
            "FAQTOTAL",
            "FAQTOTAL_CALCULATED",
            "FAQTOTAL_DIFFERENCE",
            "HAS_QC_ERROR",
            "ID",
            "SITEID",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("\nRecords with reported-versus-calculated total discrepancies:")
print(f"  Rows: {len(faq_total_discrepancy_records):,}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_total_discrepancy_records)

## 1.26. Reviewing ADNI4 quality-control flags

I am now inspecting the observed values of `HAS_QC_ERROR`.

This field is available only in ADNI4, where:

- `0` means there is no QC error, or a previous QC issue has been approved;
- `1` means the record currently has a QC error.

summarise the values, inspect all records marked `1`, and check whether those records also contain missing FAQ values, invalid scores, or other data-quality concerns.

No records will be excluded in this step.

In [ ]:
# Summarise HAS_QC_ERROR values within ADNI4.
faq_qc_error_distribution = (
    faq_clean.loc[
        faq_clean["PHASE"].eq("ADNI4"),
        "HAS_QC_ERROR"
    ]
    .astype("string")
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .rename_axis("HAS_QC_ERROR")
    .reset_index(name="row_count")
)

faq_qc_error_distribution["percentage_of_adni4"] = (
    faq_qc_error_distribution["row_count"]
    / (faq_clean["PHASE"].eq("ADNI4").sum())
    * 100
).round(2)

print("ADNI4 HAS_QC_ERROR distribution:")
display(faq_qc_error_distribution)

# Create a transparent QC flag.
faq_clean["flag_unresolved_qc_error"] = (
    faq_clean["HAS_QC_ERROR"].eq(1)
)

# Select all ADNI4 records marked with an unresolved QC error.
faq_unresolved_qc_records = (
    faq_clean.loc[
        faq_clean["flag_unresolved_qc_error"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "SOURCE",
            "LANGUAGE_CODE",
            "DD_CRF_VERSION_LABEL",
            "HAS_QC_ERROR",
            *faq_item_columns,
            "FAQTOTAL",
            "faq_missing_item_count",
            "flag_missing_total",
            "flag_item_out_of_range",
            "flag_total_out_of_range",
            "flag_total_item_discrepancy",
            "ID",
            "SITEID",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("\nRecords with HAS_QC_ERROR = 1:")
print(f"  Rows: {len(faq_unresolved_qc_records):,}")
print(
    "  Unique participants: "
    f"{faq_unresolved_qc_records['RID'].nunique(dropna=True):,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_unresolved_qc_records)

# Summarise whether QC-error records overlap with other known issues.
faq_qc_error_overlap_summary = pd.DataFrame(
    {
        "condition": [
            "HAS_QC_ERROR = 1",
            "Also has at least one missing FAQ item",
            "Also has missing FAQTOTAL",
            "Also has an item range violation",
            "Also has a total range violation",
            "Also has a total-item discrepancy",
        ],
        "row_count": [
            int(faq_clean["flag_unresolved_qc_error"].sum()),
            int(
                (
                    faq_clean["flag_unresolved_qc_error"]
                    & faq_clean["faq_missing_item_count"].gt(0)
                ).sum()
            ),
            int(
                (
                    faq_clean["flag_unresolved_qc_error"]
                    & faq_clean["flag_missing_total"]
                ).sum()
            ),
            int(
                (
                    faq_clean["flag_unresolved_qc_error"]
                    & faq_clean["flag_item_out_of_range"]
                ).sum()
            ),
            int(
                (
                    faq_clean["flag_unresolved_qc_error"]
                    & faq_clean["flag_total_out_of_range"]
                ).sum()
            ),
            int(
                (
                    faq_clean["flag_unresolved_qc_error"]
                    & faq_clean["flag_total_item_discrepancy"]
                ).sum()
            ),
        ],
    }
)

print("\nOverlap between unresolved QC errors and other FAQ issues:")
display(faq_qc_error_overlap_summary)

## 1.27. ADNI4 quality-control field interpretation

The `HAS_QC_ERROR` field is available only in ADNI4.

According to the ADNI data dictionary:

- `0` means that the record has no QC error, or that a previous QC issue was reviewed and approved;
- `1` means that the record currently has a QC error.

In the FAQ source table, all 1,527 ADNI4 records have:

```text
HAS_QC_ERROR = 0

## 1.28. Checking exact duplicates and repeated FAQ visits

I am checking whether the longitudinal FAQ table contains:

- exact duplicate records;
- more than one record for the same `RID + VISCODE2`;
- more than one record for the same `RID + VISDATE`.

Repeated records are not automatically errors. A participant may have repeated or corrected assessments, and the same visit can sometimes appear more than once with different record identifiers or values.

therefore count and display repeated records without removing anything.

In [ ]:
# Define the source-level columns used to identify exact duplicate records.
# The preserved original FAQ values are used so that duplicate detection
# reflects the source data rather than later analytical recoding.
faq_duplicate_check_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "VISDATE_ORIGINAL",
    "SOURCE",
    "SPID",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "DD_CRF_VERSION_LABEL",
    *[f"{column}_ORIGINAL" for column in faq_item_columns],
    "FAQTOTAL_ORIGINAL",
    "ID",
    "SITEID",
]

# Confirm that all duplicate-check columns exist.
missing_duplicate_columns = [
    column
    for column in faq_duplicate_check_columns
    if column not in faq_clean.columns
]

if missing_duplicate_columns:
    raise KeyError(
        "The following duplicate-check columns are missing: "
        + ", ".join(missing_duplicate_columns)
    )

# ---------------------------------------------------------------------
# Exact duplicate rows
# ---------------------------------------------------------------------

faq_exact_duplicate_mask = faq_clean.duplicated(
    subset=faq_duplicate_check_columns,
    keep=False
)

faq_exact_duplicate_records = (
    faq_clean.loc[
        faq_exact_duplicate_mask,
        faq_duplicate_check_columns,
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE_ORIGINAL", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Count duplicate rows beyond the first copy.
faq_exact_duplicate_excess_count = faq_clean.duplicated(
    subset=faq_duplicate_check_columns,
    keep="first"
).sum()

print("Exact duplicate review:")
print(
    "  Rows belonging to an exact-duplicate group: "
    f"{faq_exact_duplicate_mask.sum():,}"
)
print(
    "  Excess duplicate rows beyond the first copy: "
    f"{faq_exact_duplicate_excess_count:,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_exact_duplicate_records)


# ---------------------------------------------------------------------
# Repeated RID + VISCODE2 combinations
# ---------------------------------------------------------------------

faq_rid_viscode2_counts = (
    faq_clean
    .groupby(
        ["RID", "VISCODE2"],
        dropna=False,
    )
    .size()
    .rename("record_count")
    .reset_index()
)

faq_repeated_rid_viscode2_keys = faq_rid_viscode2_counts[
    faq_rid_viscode2_counts["record_count"] > 1
].copy()

faq_repeated_rid_viscode2_records = (
    faq_clean
    .merge(
        faq_repeated_rid_viscode2_keys[
            ["RID", "VISCODE2", "record_count"]
        ],
        on=["RID", "VISCODE2"],
        how="inner",
    )
    .sort_values(
        ["RID", "VISCODE2", "VISDATE", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nRepeated RID + VISCODE2 review:")
print(
    "  Repeated participant-visit combinations: "
    f"{len(faq_repeated_rid_viscode2_keys):,}"
)
print(
    "  Records belonging to repeated RID + VISCODE2 groups: "
    f"{len(faq_repeated_rid_viscode2_records):,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        faq_repeated_rid_viscode2_records[
            [
                "PHASE",
                "PTID",
                "RID",
                "VISCODE",
                "VISCODE2",
                "VISDATE",
                "SOURCE",
                *faq_item_columns,
                "FAQTOTAL",
                "ID",
                "SITEID",
                "record_count",
            ]
        ]
    )


# ---------------------------------------------------------------------
# Repeated RID + VISDATE combinations
# ---------------------------------------------------------------------

faq_rid_visdate_counts = (
    faq_clean
    .groupby(
        ["RID", "VISDATE"],
        dropna=False,
    )
    .size()
    .rename("record_count")
    .reset_index()
)

faq_repeated_rid_visdate_keys = faq_rid_visdate_counts[
    faq_rid_visdate_counts["record_count"] > 1
].copy()

faq_repeated_rid_visdate_records = (
    faq_clean
    .merge(
        faq_repeated_rid_visdate_keys[
            ["RID", "VISDATE", "record_count"]
        ],
        on=["RID", "VISDATE"],
        how="inner",
    )
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nRepeated RID + VISDATE review:")
print(
    "  Repeated participant-date combinations: "
    f"{len(faq_repeated_rid_visdate_keys):,}"
)
print(
    "  Records belonging to repeated RID + VISDATE groups: "
    f"{len(faq_repeated_rid_visdate_records):,}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        faq_repeated_rid_visdate_records[
            [
                "PHASE",
                "PTID",
                "RID",
                "VISCODE",
                "VISCODE2",
                "VISDATE",
                "SOURCE",
                *faq_item_columns,
                "FAQTOTAL",
                "ID",
                "SITEID",
                "record_count",
            ]
        ]
    )

## 1.29. Flagging repeated FAQ visit identifiers and dates

I found no exact duplicate FAQ records.

A small number of records share either the same `RID + VISCODE2` combination or the same `RID + VISDATE` combination. These records are not exact copies and may reflect cross-phase visit translation, repeated assessments, or harmonised date assignments.

retain all records and create separate flags for:

- repeated participant and translated-visit combinations;
- repeated participant and visit-date combinations.

These flags will support later participant-level visit selection without discarding valid longitudinal information now.

In [ ]:
# Create sets of repeated RID + VISCODE2 keys.
repeated_rid_viscode2_key_set = set(
    faq_repeated_rid_viscode2_keys[
        ["RID", "VISCODE2"]
    ].itertuples(index=False, name=None)
)

# Create sets of repeated RID + VISDATE keys.
repeated_rid_visdate_key_set = set(
    faq_repeated_rid_visdate_keys[
        ["RID", "VISDATE"]
    ].itertuples(index=False, name=None)
)

# Flag records belonging to repeated RID + VISCODE2 groups.
faq_clean["flag_repeated_rid_viscode2"] = [
    (rid, viscode2) in repeated_rid_viscode2_key_set
    for rid, viscode2 in zip(
        faq_clean["RID"],
        faq_clean["VISCODE2"],
    )
]

# Flag records belonging to repeated RID + VISDATE groups.
faq_clean["flag_repeated_rid_visdate"] = [
    (rid, visdate) in repeated_rid_visdate_key_set
    for rid, visdate in zip(
        faq_clean["RID"],
        faq_clean["VISDATE"],
    )
]

# Flag exact duplicates, although none were found.
faq_clean["flag_exact_duplicate"] = faq_exact_duplicate_mask

# Summarise the duplicate and repeated-record flags.
faq_duplicate_flag_summary = pd.DataFrame(
    {
        "flag": [
            "flag_exact_duplicate",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ],
        "row_count": [
            int(faq_clean["flag_exact_duplicate"].sum()),
            int(faq_clean["flag_repeated_rid_viscode2"].sum()),
            int(faq_clean["flag_repeated_rid_visdate"].sum()),
        ],
    }
)

faq_duplicate_flag_summary["percentage"] = (
    faq_duplicate_flag_summary["row_count"]
    / len(faq_clean)
    * 100
).round(3)

print("Duplicate and repeated-record flag summary:")
display(faq_duplicate_flag_summary)

# Display all records with either repeated-key flag.
faq_repeated_record_review = (
    faq_clean.loc[
        faq_clean[
            [
                "flag_repeated_rid_viscode2",
                "flag_repeated_rid_visdate",
            ]
        ].any(axis=1),
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "SOURCE",
            *faq_item_columns,
            "FAQTOTAL",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable"
    )
    .reset_index(drop=True)
)

print("\nAll records with repeated visit or date keys:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_repeated_record_review)

## 1.30. Decision on repeated FAQ records

No repeated records are removed during independent longitudinal FAQ cleaning.

There are no exact duplicate rows. The small number of repeated participant-visit or participant-date combinations appear to reflect cross-phase duplication, translated visit-code collisions, or harmonised date inconsistencies rather than simple duplicate entries.

All affected records are therefore retained, with the following warning flags:

- `flag_repeated_rid_viscode2`;
- `flag_repeated_rid_visdate`.

These flags do not contribute to the current provisional exclusion decision.

The repeated records will be resolved later, when one FAQ observation is selected for each participant relative to the approved cross-modality reference date. At that stage, the selection rule will consider the original visit code, phase, valid visit date, completeness, and proximity to the reference date.

## 1.31. Reviewing incomplete FAQ questionnaires

I am now examining the 243 records with at least one missing FAQ item.

These records include:

- questionnaires with all ten items missing;
- questionnaires with only some items missing;
- records affected by the undocumented `-1` placeholder;
- records that were already missing in the source data.

Before proposing exclusions, I need to distinguish fully unavailable questionnaires from partially completed questionnaires and examine how these patterns vary by ADNI phase.

No records will be removed in this step.

In [ ]:
# Classify questionnaire completeness from the cleaned analytical item fields.
faq_clean["faq_completeness_status"] = pd.Series(
    "complete",
    index=faq_clean.index,
    dtype="string",
)

faq_clean.loc[
    faq_clean["faq_missing_item_count"].eq(10),
    "faq_completeness_status",
] = "all_items_missing"

faq_clean.loc[
    faq_clean["faq_missing_item_count"].between(1, 9),
    "faq_completeness_status",
] = "partially_missing"

# Add a more detailed source-aware explanation.
faq_clean["faq_incomplete_pattern"] = pd.Series(
    "complete",
    index=faq_clean.index,
    dtype="string",
)

faq_clean.loc[
    faq_clean["flag_faq_fully_unavailable_minus_one"],
    "faq_incomplete_pattern",
] = "all_items_originally_minus_one"

faq_clean.loc[
    faq_clean["flag_faq_partially_unavailable_minus_one"],
    "faq_incomplete_pattern",
] = "some_items_originally_minus_one"

faq_clean.loc[
    faq_clean["faq_missing_item_count"].eq(10)
    & ~faq_clean["flag_faq_fully_unavailable_minus_one"],
    "faq_incomplete_pattern",
] = "all_items_missing_in_source"

faq_clean.loc[
    faq_clean["faq_missing_item_count"].between(1, 9)
    & ~faq_clean["flag_faq_partially_unavailable_minus_one"],
    "faq_incomplete_pattern",
] = "some_items_missing_in_source"

# Summarise the overall completeness patterns.
faq_completeness_summary = (
    faq_clean["faq_incomplete_pattern"]
    .value_counts(dropna=False)
    .rename_axis("faq_incomplete_pattern")
    .reset_index(name="row_count")
)

faq_completeness_summary["percentage"] = (
    faq_completeness_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("Overall FAQ completeness patterns:")
display(faq_completeness_summary)

# Summarise completeness separately by ADNI phase.
faq_completeness_by_phase = (
    faq_clean
    .groupby(
        ["PHASE", "faq_incomplete_pattern"],
        dropna=False,
    )
    .size()
    .rename("row_count")
    .reset_index()
)

phase_counts_for_completeness = (
    faq_clean
    .groupby("PHASE", dropna=False)
    .size()
)

faq_completeness_by_phase["phase_row_count"] = (
    faq_completeness_by_phase["PHASE"]
    .map(phase_counts_for_completeness)
)

faq_completeness_by_phase["percentage_within_phase"] = (
    faq_completeness_by_phase["row_count"]
    / faq_completeness_by_phase["phase_row_count"]
    * 100
).round(2)

print("\nFAQ completeness patterns by phase:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        faq_completeness_by_phase.sort_values(
            ["PHASE", "faq_incomplete_pattern"],
            kind="stable",
        )
    )

# Display every incomplete questionnaire for transparent review.
faq_incomplete_records = (
    faq_clean.loc[
        faq_clean["faq_missing_item_count"].gt(0),
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "SOURCE",
            *faq_item_columns,
            "FAQTOTAL",
            "faq_available_item_count",
            "faq_missing_item_count",
            "faq_incomplete_pattern",
            "flag_faq_fully_unavailable_minus_one",
            "flag_faq_partially_unavailable_minus_one",
            "HAS_QC_ERROR",
            "ID",
            "SITEID",
        ]
    ]
    .copy()
    .sort_values(
        ["PHASE", "RID", "VISDATE", "VISCODE2"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nAll incomplete FAQ records:")
print(f"  Rows: {len(faq_incomplete_records):,}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_incomplete_records)

## 1.32. Defining the FAQ feature used by the model

The model will use the official `FAQTOTAL` score as the FAQ-derived input feature.

`FAQTOTAL` is a continuous functional-impairment score ranging from 0 to 30. It is calculated from the ten FAQ item responses using the official scoring rules. In the current dataset, every complete questionnaire has a reported total that exactly matches the item-derived total.

therefore treat an FAQ record as model-usable only when:

- `FAQTOTAL` is present;
- `FAQTOTAL` lies within 0-30;
- the questionnaire has all ten item responses available;
- there is no unresolved QC error;
- the reported total agrees with the item-derived total.

Incomplete questionnaires will remain in the cleaned longitudinal audit table, but they will not be used as FAQ measurements. not calculate partial totals or impute missing questionnaire items.

In [ ]:
# Define whether each FAQ record contains a valid model input.
faq_clean["flag_usable_faq_measurement"] = (
    faq_clean["faq_available_item_count"].eq(10)
    & faq_clean["FAQTOTAL"].notna()
    & ~faq_clean["flag_item_out_of_range"]
    & ~faq_clean["flag_item_non_integer"]
    & ~faq_clean["flag_total_out_of_range"]
    & ~faq_clean["flag_total_non_integer"]
    & ~faq_clean["flag_total_item_discrepancy"]
    & ~faq_clean["flag_unresolved_qc_error"]
)

# Assign a transparent reason when a record is not model-usable.
faq_clean["faq_model_usability_reason"] = pd.Series(
    "usable",
    index=faq_clean.index,
    dtype="string",
)

faq_clean.loc[
    faq_clean["faq_missing_item_count"].eq(10),
    "faq_model_usability_reason",
] = "all_faq_items_missing"

faq_clean.loc[
    faq_clean["faq_missing_item_count"].between(1, 9),
    "faq_model_usability_reason",
] = "incomplete_faq_questionnaire"

faq_clean.loc[
    faq_clean["FAQTOTAL"].isna()
    & faq_clean["faq_missing_item_count"].eq(0),
    "faq_model_usability_reason",
] = "faqtotal_missing_despite_complete_items"

faq_clean.loc[
    faq_clean["flag_item_out_of_range"]
    | faq_clean["flag_item_non_integer"],
    "faq_model_usability_reason",
] = "invalid_faq_item_value"

faq_clean.loc[
    faq_clean["flag_total_out_of_range"]
    | faq_clean["flag_total_non_integer"],
    "faq_model_usability_reason",
] = "invalid_faqtotal_value"

faq_clean.loc[
    faq_clean["flag_total_item_discrepancy"],
    "faq_model_usability_reason",
] = "reported_total_disagrees_with_items"

faq_clean.loc[
    faq_clean["flag_unresolved_qc_error"],
    "faq_model_usability_reason",
] = "unresolved_adni_qc_error"

# Confirm that the reason and usability flag agree.
faq_clean.loc[
    faq_clean["flag_usable_faq_measurement"],
    "faq_model_usability_reason",
] = "usable"

# Summarise model usability.
faq_model_usability_summary = (
    faq_clean["faq_model_usability_reason"]
    .value_counts(dropna=False)
    .rename_axis("faq_model_usability_reason")
    .reset_index(name="row_count")
)

faq_model_usability_summary["percentage"] = (
    faq_model_usability_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("FAQ model-usability summary:")
display(faq_model_usability_summary)

print("\nOverall model-ready FAQ measurements:")
print(
    "  Usable FAQ rows: "
    f"{faq_clean['flag_usable_faq_measurement'].sum():,}"
)
print(
    "  Non-usable FAQ rows: "
    f"{(~faq_clean['flag_usable_faq_measurement']).sum():,}"
)

# Create the model-ready longitudinal FAQ table.
# No participant-level date selection is performed yet.
faq_model_ready_longitudinal = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_measurement"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "FAQTOTAL",
            "SOURCE",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nModel-ready longitudinal FAQ table:")
print(f"  Rows: {len(faq_model_ready_longitudinal):,}")
print(
    "  Unique participants: "
    f"{faq_model_ready_longitudinal['RID'].nunique():,}"
)

display(faq_model_ready_longitudinal.head(10))

## 1.33. Correcting structural missingness in the ADNI4 QC flag

The first model-ready FAQ table incorrectly retained only ADNI4 observations.

`HAS_QC_ERROR` was collected only in ADNI4. In earlier phases, its missing values are structural and do not indicate a quality-control failure. However, the previous Boolean expression preserved these values as indeterminate (`<NA>`), preventing otherwise valid records from being selected.

reconstruct the unresolved-QC flag so that:

- `HAS_QC_ERROR = 1` is treated as an unresolved QC error;
- `HAS_QC_ERROR = 0` is treated as no unresolved QC error;
- structural missingness outside ADNI4 is treated as not applicable and does not invalidate the FAQ measurement.

then rebuild the FAQ model-usability flag and longitudinal model-ready table.

In [ ]:
# Reconstruct the unresolved-QC flag as an ordinary Boolean variable.
# Missing HAS_QC_ERROR values belong to phases where the field was not collected,
# so they must not be interpreted as QC failures.
faq_clean["flag_unresolved_qc_error"] = (
    faq_clean["HAS_QC_ERROR"]
    .eq(1)
    .fillna(False)
    .astype(bool)
)

# Rebuild the model-usability flag.
faq_clean["flag_usable_faq_measurement"] = (
    faq_clean["faq_available_item_count"].eq(10)
    & faq_clean["FAQTOTAL"].notna()
    & ~faq_clean["flag_item_out_of_range"].fillna(False)
    & ~faq_clean["flag_item_non_integer"].fillna(False)
    & ~faq_clean["flag_total_out_of_range"].fillna(False)
    & ~faq_clean["flag_total_non_integer"].fillna(False)
    & ~faq_clean["flag_total_item_discrepancy"].fillna(False)
    & ~faq_clean["flag_unresolved_qc_error"]
).astype(bool)

# Rebuild the usability reason from the corrected flag.
faq_clean["faq_model_usability_reason"] = pd.Series(
    "usable",
    index=faq_clean.index,
    dtype="string",
)

faq_clean.loc[
    faq_clean["faq_missing_item_count"].eq(10),
    "faq_model_usability_reason",
] = "all_faq_items_missing"

faq_clean.loc[
    faq_clean["faq_missing_item_count"].between(1, 9),
    "faq_model_usability_reason",
] = "incomplete_faq_questionnaire"

faq_clean.loc[
    faq_clean["flag_unresolved_qc_error"],
    "faq_model_usability_reason",
] = "unresolved_adni_qc_error"

# Ensure every record labelled usable actually passes the corrected flag.
faq_clean.loc[
    ~faq_clean["flag_usable_faq_measurement"]
    & faq_clean["faq_model_usability_reason"].eq("usable"),
    "faq_model_usability_reason",
] = "other_nonusable_reason"

# Recreate the summary.
faq_model_usability_summary = (
    faq_clean["faq_model_usability_reason"]
    .value_counts(dropna=False)
    .rename_axis("faq_model_usability_reason")
    .reset_index(name="row_count")
)

faq_model_usability_summary["percentage"] = (
    faq_model_usability_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("Corrected FAQ model-usability summary:")
display(faq_model_usability_summary)

print("\nCorrected model-ready FAQ measurements:")
print(
    "  Usable FAQ rows: "
    f"{faq_clean['flag_usable_faq_measurement'].sum():,}"
)
print(
    "  Non-usable FAQ rows: "
    f"{(~faq_clean['flag_usable_faq_measurement']).sum():,}"
)

# Rebuild the model-ready longitudinal table.
faq_model_ready_longitudinal = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_measurement"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "FAQTOTAL",
            "SOURCE",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

print("\nCorrected model-ready longitudinal FAQ table:")
print(f"  Rows: {len(faq_model_ready_longitudinal):,}")
print(
    "  Unique participants: "
    f"{faq_model_ready_longitudinal['RID'].nunique():,}"
)

# Confirm that usable rows are represented across every ADNI phase.
faq_model_ready_phase_summary = (
    faq_model_ready_longitudinal["PHASE"]
    .value_counts()
    .rename_axis("PHASE")
    .reset_index(name="usable_row_count")
)

print("\nUsable FAQ rows by phase:")
display(faq_model_ready_phase_summary)

display(faq_model_ready_longitudinal.head(10))

## 1.34. Checking missing visit dates among usable FAQ measurements

The FAQ value itself is usable only when the questionnaire is complete and the official total is valid. However, later participant-level selection will also require a valid visit date so that the FAQ assessment can be aligned to the approved reference date.

therefore check:

- how many usable FAQ measurements have a missing `VISDATE`;
- which participants and visits are affected;
- whether a usable record with a valid date exists for the same participant and visit code.

No records will be removed yet.

In [ ]:
# Identify usable FAQ measurements with a missing visit date.
faq_clean["flag_usable_faq_missing_visdate"] = (
    faq_clean["flag_usable_faq_measurement"]
    & faq_clean["VISDATE"].isna()
)

usable_missing_visdate_count = int(
    faq_clean["flag_usable_faq_missing_visdate"].sum()
)

print("Usable FAQ measurements with missing VISDATE:")
print(f"  Rows: {usable_missing_visdate_count:,}")
print(
    "  Unique participants: "
    f"{faq_clean.loc[faq_clean['flag_usable_faq_missing_visdate'], 'RID'].nunique():,}"
)

# Display affected records.
faq_usable_missing_visdate_records = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_missing_visdate"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE_ORIGINAL",
            "VISDATE",
            "SOURCE",
            *faq_item_columns,
            "FAQTOTAL",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nAffected usable FAQ records:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(faq_usable_missing_visdate_records)

# Check whether the same participant and translated visit code
# also has another usable FAQ record with a valid visit date.
valid_date_usable_keys = set(
    faq_clean.loc[
        faq_clean["flag_usable_faq_measurement"]
        & faq_clean["VISDATE"].notna(),
        ["RID", "VISCODE2"],
    ].itertuples(index=False, name=None)
)

faq_usable_missing_visdate_records[
    "same_rid_viscode2_has_valid_date_record"
] = [
    (rid, viscode2) in valid_date_usable_keys
    for rid, viscode2 in zip(
        faq_usable_missing_visdate_records["RID"],
        faq_usable_missing_visdate_records["VISCODE2"],
    )
]

print("\nWhether an alternative valid-date record exists:")
display(
    faq_usable_missing_visdate_records[
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "FAQTOTAL",
            "same_rid_viscode2_has_valid_date_record",
        ]
    ]
)

## 1.35. Recovering missing FAQ visit dates from the authoritative registry

Four otherwise usable FAQ measurements have no visit date.

One of them, RID 830 at `m60`, already has another FAQ record with the same score and a valid date. The remaining three records may have dates available in the ADNI `REGISTRY` table.

inspect the registry records for these four participants and compare their visit identifiers. not fill any dates yet; this step only identifies reliable candidate matches.

In [ ]:
from pathlib import Path
import pandas as pd

# Locate the downloaded REGISTRY file.
registry_candidates = list(
    Path(
        "/content/drive/MyDrive/adni_mri/adni_non_imaging/raw/"
        "Cohort, dates and source-of-truth tables"
    ).glob("REGISTRY*.csv")
)

if not registry_candidates:
    raise FileNotFoundError(
        "No REGISTRY CSV was found in the expected source-of-truth directory."
    )

if len(registry_candidates) > 1:
    print("Multiple REGISTRY files were found:")
    for path in registry_candidates:
        print(f"  {path}")

registry_path = registry_candidates[0]

print(f"Using REGISTRY file:\n  {registry_path}")

# Load the registry without modifying the source file.
registry_raw = pd.read_csv(
    registry_path,
    low_memory=False,
)

print("\nREGISTRY shape:")
print(f"  Rows: {len(registry_raw):,}")
print(f"  Columns: {registry_raw.shape[1]:,}")

print("\nREGISTRY columns:")
print(registry_raw.columns.tolist())

# The four usable FAQ records whose source visit date is missing.
missing_date_rids = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_missing_visdate"],
        "RID",
    ]
    .dropna()
    .astype("Int64")
    .unique()
    .tolist()
)

# Select every registry row for the affected participants.
registry_missing_date_candidates = (
    registry_raw.loc[
        pd.to_numeric(
            registry_raw["RID"],
            errors="coerce",
        ).isin(missing_date_rids)
    ]
    .copy()
)

# Display the most relevant visit and date columns that actually exist.
preferred_registry_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "VISDATE",
    "USERDATE",
    "USERDATE2",
    "PTSTATUS",
    "RGSTATUS",
]

available_registry_columns = [
    column
    for column in preferred_registry_columns
    if column in registry_missing_date_candidates.columns
]

# Sort using whichever identifiers are available.
registry_sort_columns = [
    column
    for column in ["RID", "EXAMDATE", "VISDATE", "VISCODE2", "VISCODE"]
    if column in registry_missing_date_candidates.columns
]

if registry_sort_columns:
    registry_missing_date_candidates = (
        registry_missing_date_candidates
        .sort_values(
            registry_sort_columns,
            kind="stable",
            na_position="last",
        )
    )

print("\nRegistry records for the four affected participants:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(
        registry_missing_date_candidates[
            available_registry_columns
        ].reset_index(drop=True)
    )

# Show the undated FAQ records again beside their identifiers.
print("\nUndated usable FAQ records requiring comparison:")

display(
    faq_usable_missing_visdate_records[
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "FAQTOTAL",
            "same_rid_viscode2_has_valid_date_record",
        ]
    ]
)

## 1.36. Distinguishing valid FAQ values from date-alignable measurements

The registry review did not provide an authoritative examination date for any of the four undated FAQ records.

Although registry `USERDATE` and `USERDATE2` values are available for some of these records, these are administrative data-entry or update dates rather than assessment dates. therefore not use them to replace `VISDATE`.

retain two separate usability indicators:

- `flag_usable_faq_measurement`: the questionnaire contains a valid FAQ value;
- `flag_date_alignable_faq_measurement`: the questionnaire contains a valid FAQ value and a valid assessment date.

The four undated records will remain in the cleaned audit table but will not be eligible for later reference-date matching.

In [ ]:
# Define whether a valid FAQ measurement can also be aligned by date.
faq_clean["flag_date_alignable_faq_measurement"] = (
    faq_clean["flag_usable_faq_measurement"]
    & faq_clean["VISDATE"].notna()
).astype(bool)

# Assign a transparent date-alignment status.
faq_clean["faq_date_alignment_status"] = pd.Series(
    "not_value_usable",
    index=faq_clean.index,
    dtype="string",
)

faq_clean.loc[
    faq_clean["flag_usable_faq_measurement"]
    & faq_clean["VISDATE"].isna(),
    "faq_date_alignment_status",
] = "valid_faq_but_missing_visit_date"

faq_clean.loc[
    faq_clean["flag_date_alignable_faq_measurement"],
    "faq_date_alignment_status",
] = "valid_and_date_alignable"

# Add a specific note for the undated cross-phase duplicate.
faq_clean["flag_undated_duplicate_with_dated_equivalent"] = False

faq_clean.loc[
    faq_clean["RID"].eq(830)
    & faq_clean["PHASE"].eq("ADNIGO")
    & faq_clean["VISCODE2"].eq("m60")
    & faq_clean["VISDATE"].isna()
    & faq_clean["flag_usable_faq_measurement"],
    "flag_undated_duplicate_with_dated_equivalent",
] = True

# Summarise the distinction between value usability and date alignment.
faq_alignment_summary = pd.DataFrame(
    {
        "category": [
            "Valid FAQ measurements",
            "Valid and date-alignable FAQ measurements",
            "Valid FAQ measurements missing VISDATE",
            "Non-usable FAQ questionnaires",
        ],
        "row_count": [
            int(faq_clean["flag_usable_faq_measurement"].sum()),
            int(faq_clean["flag_date_alignable_faq_measurement"].sum()),
            int(
                (
                    faq_clean["flag_usable_faq_measurement"]
                    & faq_clean["VISDATE"].isna()
                ).sum()
            ),
            int((~faq_clean["flag_usable_faq_measurement"]).sum()),
        ],
    }
)

faq_alignment_summary["percentage_of_all_rows"] = (
    faq_alignment_summary["row_count"]
    / len(faq_clean)
    * 100
).round(2)

print("FAQ value-usability and date-alignment summary:")
display(faq_alignment_summary)

# Create the date-alignable longitudinal table that will later be used
# for matching FAQ observations to participant reference dates.
faq_date_alignable_longitudinal = (
    faq_clean.loc[
        faq_clean["flag_date_alignable_faq_measurement"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "FAQTOTAL",
            "SOURCE",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("\nDate-alignable longitudinal FAQ table:")
print(f"  Rows: {len(faq_date_alignable_longitudinal):,}")
print(
    "  Unique participants: "
    f"{faq_date_alignable_longitudinal['RID'].nunique():,}"
)

# Display the four value-valid but undated records retained only for audit.
faq_valid_but_undated = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_measurement"]
        & ~faq_clean["flag_date_alignable_faq_measurement"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "FAQTOTAL",
            "ID",
            "flag_undated_duplicate_with_dated_equivalent",
            "faq_date_alignment_status",
        ]
    ]
    .copy()
    .sort_values(["RID", "ID"], kind="stable")
    .reset_index(drop=True)
)

print("\nValid FAQ records retained for audit but unavailable for date matching:")
display(faq_valid_but_undated)

## 1.37. Saving the final FAQ outputs

I am saving the completed FAQ preprocessing outputs.

The cleaned longitudinal audit table retains all source records and QC flags. The model-ready table keeps only valid FAQ measurements with a valid visit date, because these are the records that can later be aligned to the participant reference date.

I am also saving compact QC summary tables for reproducibility.

In [ ]:
from pathlib import Path
import pandas as pd

# Define final output paths.
faq_clean_output_path = (
    processed_dir / "faq_clean_longitudinal_interim.csv"
)

faq_model_ready_output_path = (
    processed_dir / "faq_model_ready_dated_longitudinal.csv"
)

faq_qc_summary_output_path = (
    qc_dir / "faq_preprocessing_summary.csv"
)

faq_nonusable_output_path = (
    qc_dir / "faq_nonusable_records.csv"
)

faq_repeated_output_path = (
    qc_dir / "faq_repeated_visit_date_records.csv"
)

# Ensure output directories exist.
processed_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

# Create the final date-alignable model-ready table directly from faq_clean.
faq_model_ready_dated = (
    faq_clean.loc[
        faq_clean["flag_usable_faq_measurement"]
        & faq_clean["VISDATE"].notna(),
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "FAQTOTAL",
            "SOURCE",
            "ID",
            "SITEID",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Create one compact preprocessing summary.
faq_preprocessing_summary = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "unique_participants_raw",
            "usable_faq_rows",
            "usable_faq_unique_participants",
            "model_ready_dated_rows",
            "model_ready_dated_unique_participants",
            "all_items_missing_rows",
            "partially_incomplete_rows",
            "original_minus_one_affected_rows",
            "usable_rows_missing_visdate",
            "exact_duplicate_rows",
            "repeated_rid_viscode2_rows",
            "repeated_rid_visdate_rows",
            "unresolved_qc_error_rows",
            "total_item_discrepancy_rows",
        ],
        "value": [
            len(faq_clean),
            faq_clean["RID"].nunique(dropna=True),
            int(faq_clean["flag_usable_faq_measurement"].sum()),
            faq_clean.loc[
                faq_clean["flag_usable_faq_measurement"],
                "RID",
            ].nunique(dropna=True),
            len(faq_model_ready_dated),
            faq_model_ready_dated["RID"].nunique(dropna=True),
            int(faq_clean["faq_missing_item_count"].eq(10).sum()),
            int(faq_clean["faq_missing_item_count"].between(1, 9).sum()),
            int(faq_clean["flag_any_faq_minus_one"].sum()),
            int(
                (
                    faq_clean["flag_usable_faq_measurement"]
                    & faq_clean["VISDATE"].isna()
                ).sum()
            ),
            int(faq_clean["flag_exact_duplicate"].sum()),
            int(faq_clean["flag_repeated_rid_viscode2"].sum()),
            int(faq_clean["flag_repeated_rid_visdate"].sum()),
            int(faq_clean["flag_unresolved_qc_error"].sum()),
            int(faq_clean["flag_total_item_discrepancy"].sum()),
        ],
    }
)

# Save non-usable FAQ records for transparent QC review.
faq_nonusable_records = (
    faq_clean.loc[
        ~faq_clean["flag_usable_faq_measurement"]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2"],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

# Save all repeated visit/date records for later reference-date resolution.
faq_repeated_records = (
    faq_clean.loc[
        faq_clean["flag_repeated_rid_viscode2"]
        | faq_clean["flag_repeated_rid_visdate"]
    ]
    .copy()
    .sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

# Save all outputs.
faq_clean.to_csv(
    faq_clean_output_path,
    index=False,
)

faq_model_ready_dated.to_csv(
    faq_model_ready_output_path,
    index=False,
)

faq_preprocessing_summary.to_csv(
    faq_qc_summary_output_path,
    index=False,
)

faq_nonusable_records.to_csv(
    faq_nonusable_output_path,
    index=False,
)

faq_repeated_records.to_csv(
    faq_repeated_output_path,
    index=False,
)

# Confirm saved files and final counts.
print("FAQ preprocessing completed and saved.\n")

print("Clean longitudinal audit table:")
print(f"  Path: {faq_clean_output_path}")
print(f"  Rows: {len(faq_clean):,}")

print("\nModel-ready dated longitudinal table:")
print(f"  Path: {faq_model_ready_output_path}")
print(f"  Rows: {len(faq_model_ready_dated):,}")
print(
    "  Unique participants: "
    f"{faq_model_ready_dated['RID'].nunique():,}"
)

print("\nQC summary:")
print(f"  Path: {faq_qc_summary_output_path}")

print("\nNon-usable record log:")
print(f"  Path: {faq_nonusable_output_path}")
print(f"  Rows: {len(faq_nonusable_records):,}")

print("\nRepeated-record log:")
print(f"  Path: {faq_repeated_output_path}")
print(f"  Rows: {len(faq_repeated_records):,}")

display(faq_preprocessing_summary)